In [1]:
from ..utils.utils_models import *
# from scipy.optimize import minimize

ImportError: attempted relative import with no known parent package

# Get some initial params

In [9]:
EJ = 4
EC = 4/5.9
EL = 4/29.2

qubit_level = 30


def get_shift_accurate(ele,omega_i, omega_j, omega_r):
    return abs(ele)**2 / (omega_j-omega_i-omega_r) - abs(ele)**2 / (omega_i-omega_j-omega_r)


qbt = scqubits.Fluxonium(EJ=EJ,EC=EC,EL=EL,flux=0,cutoff=110,truncated_dim=qubit_level)
evals = qbt.eigenvals(qubit_level)
elements = qbt.matrixelement_table('n_operator',evals_count = qubit_level)

def shift_diff(x):
    Er = x[0]
    shifts_from_zero = [get_shift_accurate(elements[1,ql2],evals[ql2],evals[0],Er) for ql2 in range(qubit_level)] 
    shift_from_zero = sum(shifts_from_zero)
    shifts_from_two = [get_shift_accurate(elements[2,ql2],evals[ql2],evals[2],Er) for ql2 in range(qubit_level)] 
    shift_from_two = sum(shifts_from_two)
    return abs(shift_from_zero-shift_from_two)

initial_guess = [6.375]

# Call the optimizer
result = minimize(shift_diff, initial_guess, method='L-BFGS-B')

print("Result:", result)
print("Optimal solution:", result.x)
print("Objective value:", result.fun)


Result:   message: ABNORMAL_TERMINATION_IN_LNSRCH
  success: False
   status: 2
      fun: 1.2691810158393935e-08
        x: [ 6.362e+00]
      nit: 4
      jac: [ 4.312e+00]
     nfev: 106
     njev: 53
 hess_inv: <1x1 LbfgsInvHessProduct with dtype=float64>
Optimal solution: [6.36228152]
Objective value: 1.2691810158393935e-08


In [10]:
def shift_diff(x):
    Ej = x[0]
    tune_tmon = scqubits.Transmon(
        EJ=Ej,
        EC=0.2,
        ng=0.0,
        ncut=10,
        truncated_dim = 4
        )
    evals = tune_tmon.eigenvals()
    return abs(evals[1]-evals[0] - 6.36228152)

initial_guess = np.array([34])

bounds = [(20, 50)]

result = minimize(shift_diff, initial_guess, bounds=bounds, method='L-BFGS-B')

print("Optimal solution:", result.x)
print("Optimal value:", result.fun)

Optimal solution: [26.96918998]
Optimal value: 7.838671933768637e-10


#  Let's test our rabi frequency equation first by doing qutip evo

In [12]:
def detuned_rabi_period(amp, element, detunning):
    return 2*np.pi / np.sqrt(
        (amp*element)**2
        + (2*np.pi*detunning)**2
    )


In [ ]:
Ej_list = np.linspace(26,28,100)
amp_list  = np.linspace(5e-3,0.1e-1, 500)

me0001 = []
me1011 = []
me2021 = []
zero_period = []
one_period = []
two_period = []
qbt_frequency_tmon0 = []
qbt_frequency_tmon1 = []


fluxonium =scqubits.Fluxonium(EJ=EJ,EC=EC,EL=EL,flux=0,cutoff=110,truncated_dim=qubit_level)
for Ej_transmon in tqdm(Ej_list):
    transmon = scqubits.Transmon(
        EJ=Ej_transmon,
        EC=0.2,
        ng=0.0,
        ncut=10,
        truncated_dim = 4
        )
    system = FluxoniumTransmonSystem(
        fluxonium  = fluxonium,
        transmon = transmon,
        computaional_states = '1,2',
        g_strength = 0.15,
        )
    
    op = system.hilbertspace.op_in_dressed_eigenbasis(transmon.n_operator)
    me0001.append(abs(op[system.product_to_dressed[(0,0)],  system.product_to_dressed[(0,1)]]))
    me1011.append(abs(op[system.product_to_dressed[(1,0)],  system.product_to_dressed[(1,1)]]))
    me2021.append(abs(op[system.product_to_dressed[(2,0)],  system.product_to_dressed[(2,1)]]))

    zero_period.append([])
    one_period.append([])
    two_period.append([])
    
    for amp in amp_list:
        zero_period[-1].append( np.pi/ (amp *abs(op[system.product_to_dressed[(0,0)],  system.product_to_dressed[(0,1)]])  ))
        one_period[-1].append(
            2*np.pi / np.sqrt( 
                (amp * abs(op[system.product_to_dressed[(1,0)],  system.product_to_dressed[(1,1)]]))**2 +\
            (2*np.pi*(dressed_ener(1,1)-dressed_ener(1,0)) - 2*np.pi*(dressed_ener(0,1)-dressed_ener(0,0)))**2
            )
        )
        two_period[-1].append(
            2*np.pi / np.sqrt( 
                (amp *abs(op[system.product_to_dressed[(2,0)],  system.product_to_dressed[(2,1)]]))**2 +\
            (2*np.pi*(dressed_ener(2,1)-dressed_ener(2,0)) - 2*np.pi*(dressed_ener(0,1)-dressed_ener(0,0)))**2)
        )

    qbt_frequency_tmon0.append(2*np.pi*(dressed_ener(2,0)-dressed_ener(1,0)))
    qbt_frequency_tmon1.append(2*np.pi*(dressed_ener(2,1)-dressed_ener(1,1)))